# SiPeka — Step 5: Exploratory Data Analysis (EDA) Deskriptif

Notebook ini menganalisis dataset bersih hasil Step 4 (`data/processed/features_v1`) untuk menjawab tiga pertanyaan analitik dari PRD:

1. **Pertanyaan Analitik 1:** Bagaimana perubahan volume pengangkutan sampah berdasarkan waktu dan wilayah?
2. **Pertanyaan Analitik 2:** Bagaimana pola antar periode (2018 vs 2020 vs 2021, pola hari dalam minggu, serta musim hujan vs kemarau)?
3. **Pertanyaan Analitik 4:** Wilayah dan `titik_lokasi` mana yang memiliki volume pengangkutan tertinggi?

**Prinsip pemrosesan:** seluruh agregasi dilakukan di **PySpark**, lalu hanya tabel agregat kecil (atau sampel visualisasi terbatas) yang dikonversi ke **pandas** untuk visualisasi **Plotly** dan diekspor ke `docs/figures/step5_*.html` & `.json`.

## 0. Impor, SparkSession, & Pemuatan Dataset

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

ROOT_PROYEK = Path(os.environ.get("PYTHONPATH", "/workspace"))
if not ROOT_PROYEK.exists():
    ROOT_PROYEK = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_PROYEK) not in sys.path:
    sys.path.insert(0, str(ROOT_PROYEK))

import pandas as pd
import plotly.express as px
from IPython.display import display
from pyspark.sql import functions as F

from config.settings import DATA_PROCESSED_DIR, FIGURES_DIR, get_settings
from src.utils.spark_session import get_spark, tampilkan_info

SETTINGS = get_settings()
SETTINGS.pastikan_folder_lokal()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SPARK = get_spark("SiPeka-Step5-EDA")
tampilkan_info(SPARK)

FEATURES_V1 = os.environ.get(
    "SIPEKA_FEATURES_V1", str(DATA_PROCESSED_DIR / "features_v1")
)
df = SPARK.read.parquet(Path(FEATURES_V1).resolve().as_uri()).cache()

total_baris = df.count()
total_outlier = df.filter(F.col("is_outlier") == 1).count()
print(f"Total baris bersih : {total_baris:,}")
print(f"Baris is_outlier=1 : {total_outlier:,} ({total_outlier / total_baris * 100:.3f}%)")

App Name       : SiPeka-Step5-EDA
Spark Version  : 3.5.3
Master         : local[4]
Driver Memory  : 3g
Default FS     : file:///


Total baris bersih : 1,068,562
Baris is_outlier=1 : 1,068 (0.100%)


In [2]:
def simpan_figure(fig, nama_dasar: str) -> None:
    """Simpan figure Plotly ke HTML dan JSON lalu tampilkan."""
    fig.write_html(str(FIGURES_DIR / f"{nama_dasar}.html"))
    fig.write_json(str(FIGURES_DIR / f"{nama_dasar}.json"))
    fig.show()


# Dataset tanpa outlier ekstrem (is_outlier == 0) dipakai untuk membaca pola normal
# tanpa terdistorsi anomali ekstrem (jutaan m³ di data mentah).
df_normal = df.filter(F.col("is_outlier") == 0)

ringkasan_tahun = (
    df.groupBy("tahun")
    .agg(
        F.count("*").alias("jumlah_baris"),
        F.countDistinct("titik_lokasi").alias("titik_unik"),
        F.sum("volume_sampah_perhari").alias("total_volume_raw"),
        F.mean("volume_sampah_perhari").alias("rata_volume_raw"),
        F.max("volume_sampah_perhari").alias("maks_volume_raw"),
    )
    .join(
        df_normal.groupBy("tahun").agg(
            F.sum("volume_sampah_perhari").alias("total_volume_normal"),
            F.mean("volume_sampah_perhari").alias("rata_volume_normal"),
        ),
        on="tahun",
        how="left",
    )
    .orderBy("tahun")
    .toPandas()
)
display(ringkasan_tahun)

,tahun,jumlah_baris,titik_unik,total_volume_raw,rata_volume_raw,maks_volume_raw,total_volume_normal,rata_volume_normal
0,2018,251490,2055,4.739980e+09,18847.588033,156249999.0,366811.4774,1.459667
1,2020,290301,1851,8.438629e+05,2.906855,2358.0,760925.3600,2.628476
2,2021,526771,1771,8.025491e+05,1.523526,1176.0,789941.1300,1.499785


## 1. Tren Volume Harian & Bulanan per Tahun (2018 vs 2020 vs 2021)

Untuk menjaga keterbacaan tren operasional harian/bulanan, tren normal dihitung pada `is_outlier == 0` (`<= P99.9`), sedangkan lonjakan ekstrem dianalisis terpisah di Bagian 4.

In [3]:
tren_harian = (
    df_normal.groupBy("tahun", "tanggal")
    .agg(
        F.sum("volume_sampah_perhari").alias("total_harian"),
        F.mean("volume_sampah_perhari").alias("rata_harian"),
    )
    .orderBy("tanggal")
    .toPandas()
)
tren_harian["tahun_str"] = tren_harian["tahun"].astype(str)

fig_harian = px.line(
    tren_harian,
    x="tanggal",
    y="total_harian",
    color="tahun_str",
    title="Tren Total Volume Pengangkutan Harian (is_outlier = 0)",
    labels={
        "tanggal": "Tanggal",
        "total_harian": "Total volume harian (m³)",
        "tahun_str": "Tahun",
    },
)
simpan_figure(fig_harian, "step5_tren_harian")

tren_bulanan = (
    df_normal.groupBy("tahun", "bulan")
    .agg(
        F.sum("volume_sampah_perhari").alias("total_bulanan"),
        F.mean("volume_sampah_perhari").alias("rata_bulanan"),
        F.count("*").alias("jumlah_observasi"),
    )
    .orderBy("tahun", "bulan")
    .toPandas()
)
tren_bulanan["tahun_str"] = tren_bulanan["tahun"].astype(str)

fig_bulanan_total = px.line(
    tren_bulanan,
    x="bulan",
    y="total_bulanan",
    color="tahun_str",
    markers=True,
    title="Perbandingan Total Volume Bulanan: 2018 vs 2020 vs 2021 (is_outlier = 0)",
    labels={
        "bulan": "Bulan",
        "total_bulanan": "Total volume bulanan (m³)",
        "tahun_str": "Tahun",
    },
)
simpan_figure(fig_bulanan_total, "step5_tren_bulanan_total")

fig_bulanan_rata = px.line(
    tren_bulanan,
    x="bulan",
    y="rata_bulanan",
    color="tahun_str",
    markers=True,
    title="Perbandingan Rata-rata Volume Harian per Titik per Bulan (is_outlier = 0)",
    labels={
        "bulan": "Bulan",
        "rata_bulanan": "Rata-rata volume per baris (m³)",
        "tahun_str": "Tahun",
    },
)
simpan_figure(fig_bulanan_rata, "step5_tren_bulanan_rata")

puncak_bulanan = tren_bulanan.loc[tren_bulanan["total_bulanan"].idxmax()]
terendah_bulanan = tren_bulanan.loc[tren_bulanan["total_bulanan"].idxmin()]
print(
    f"Bulan tertinggi (normal): {int(puncak_bulanan['tahun'])}-{int(puncak_bulanan['bulan']):02d} "
    f"= {puncak_bulanan['total_bulanan']:,.2f} m³ | "
    f"terendah: {int(terendah_bulanan['tahun'])}-{int(terendah_bulanan['bulan']):02d} "
    f"= {terendah_bulanan['total_bulanan']:,.2f} m³"
)

Bulan tertinggi (normal): 2021-01 = 150,173.47 m³ | terendah: 2018-01 = 11,276.50 m³


## 2. Pola Wilayah, Kecamatan, Heatmap Bulan x Wilayah, & Hari dalam Minggu

In [4]:
agg_wilayah = (
    df_normal.groupBy("wilayah")
    .agg(
        F.sum("volume_sampah_perhari").alias("total_volume"),
        F.mean("volume_sampah_perhari").alias("rata_volume"),
        F.countDistinct("titik_lokasi").alias("jumlah_titik"),
    )
    .orderBy(F.desc("total_volume"))
    .toPandas()
)
display(agg_wilayah)

fig_wilayah = px.bar(
    agg_wilayah,
    x="wilayah",
    y="total_volume",
    text_auto=".2s",
    title="Total Volume Pengangkutan per Wilayah (is_outlier = 0)",
    labels={"wilayah": "Wilayah", "total_volume": "Total volume (m³)"},
)
simpan_figure(fig_wilayah, "step5_volume_per_wilayah")

agg_kecamatan = (
    df_normal.groupBy("wilayah", "kecamatan")
    .agg(
        F.sum("volume_sampah_perhari").alias("total_volume"),
        F.mean("volume_sampah_perhari").alias("rata_volume"),
    )
    .orderBy(F.desc("total_volume"))
    .limit(15)
    .toPandas()
)
fig_kec = px.bar(
    agg_kecamatan,
    x="total_volume",
    y="kecamatan",
    color="wilayah",
    orientation="h",
    title="Top 15 Kecamatan dengan Total Volume Tertinggi (is_outlier = 0)",
    labels={"total_volume": "Total volume (m³)", "kecamatan": "Kecamatan"},
)
fig_kec.update_layout(yaxis={"categoryorder": "total ascending"})
simpan_figure(fig_kec, "step5_top15_kecamatan")

heatmap_df = (
    df_normal.groupBy("wilayah", "bulan")
    .agg(F.mean("volume_sampah_perhari").alias("rata_volume"))
    .toPandas()
    .pivot(index="wilayah", columns="bulan", values="rata_volume")
    .fillna(0.0)
)
fig_heatmap = px.imshow(
    heatmap_df,
    aspect="auto",
    color_continuous_scale="YlOrRd",
    title="Heatmap Rata-rata Volume Harian: Wilayah x Bulan (is_outlier = 0)",
    labels={"x": "Bulan", "y": "Wilayah", "color": "Rata-rata (m³)"},
)
simpan_figure(fig_heatmap, "step5_heatmap_bulan_wilayah")

# Pola hari dalam minggu (Spark dayofweek: 1=Minggu .. 7=Sabtu)
nama_hari = {
    1: "1. Minggu",
    2: "2. Senin",
    3: "3. Selasa",
    4: "4. Rabu",
    5: "5. Kamis",
    6: "6. Jumat",
    7: "7. Sabtu",
}
pola_hari = (
    df_normal.groupBy("hari_dalam_minggu")
    .agg(
        F.mean("volume_sampah_perhari").alias("rata_volume"),
        F.sum("volume_sampah_perhari").alias("total_volume"),
        F.count("*").alias("n"),
    )
    .orderBy("hari_dalam_minggu")
    .toPandas()
)
pola_hari["hari"] = pola_hari["hari_dalam_minggu"].map(nama_hari)
display(pola_hari)

fig_hari = px.bar(
    pola_hari,
    x="hari",
    y="rata_volume",
    title="Pola Rata-rata Volume Berdasarkan Hari dalam Minggu (is_outlier = 0)",
    labels={"hari": "Hari", "rata_volume": "Rata-rata volume (m³)"},
)
simpan_figure(fig_hari, "step5_pola_hari_minggu")

# Perbandingan musim hujan (Okt-Apr) vs kemarau (Mei-Sep)
pola_musim = (
    df_normal.groupBy("is_musim_hujan")
    .agg(
        F.count("*").alias("n"),
        F.mean("volume_sampah_perhari").alias("rata_volume"),
        F.sum("volume_sampah_perhari").alias("total_volume"),
    )
    .toPandas()
)
pola_musim["musim"] = pola_musim["is_musim_hujan"].map(
    {1: "Musim Hujan (Okt-Apr)", 0: "Musim Kemarau (Mei-Sep)"}
)
display(pola_musim)

,wilayah,total_volume,rata_volume,jumlah_titik
0,JAKARTA BARAT,670378.5915,2.334553,1074
1,JAKARTA SELATAN,386779.4558,2.152638,427
2,JAKARTA PUSAT,335901.2152,2.846669,185
3,JAKARTA UTARA,269020.8165,0.908962,459
4,JAKARTA TIMUR,255597.8884,1.369037,849


,hari_dalam_minggu,rata_volume,total_volume,n,hari
0,1,1.715968,254532.9396,148332,1. Minggu
1,2,1.765393,266482.5128,150948,2. Senin
2,3,1.910881,286091.4412,149717,3. Selasa
3,4,1.913500,291395.4517,152284,4. Rabu
4,5,1.785043,281724.4744,157825,5. Kamis
5,6,1.707067,262577.5618,153818,6. Jumat
6,7,1.778311,274873.5859,154570,7. Sabtu


,is_musim_hujan,n,rata_volume,total_volume,musim
0,1,615148,1.917054,1.179272e+06,Musim Hujan (Okt-Apr)
1,0,452346,1.632393,7.384063e+05,Musim Kemarau (Mei-Sep)


## 3. Top 10 `titik_lokasi` dengan Volume Tertinggi

In [5]:
top10_total = (
    df_normal.groupBy("titik_lokasi", "wilayah", "kecamatan")
    .agg(
        F.sum("volume_sampah_perhari").alias("total_volume"),
        F.mean("volume_sampah_perhari").alias("rata_harian"),
        F.count("*").alias("jumlah_hari"),
    )
    .orderBy(F.desc("total_volume"))
    .limit(10)
    .toPandas()
)
print("Top 10 titik berdasarkan TOTAL volume (is_outlier = 0):")
display(top10_total)

fig_top_total = px.bar(
    top10_total,
    x="total_volume",
    y="titik_lokasi",
    color="wilayah",
    orientation="h",
    title="Top 10 Titik Lokasi Berdasarkan Total Volume (is_outlier = 0)",
    labels={"total_volume": "Total volume (m³)", "titik_lokasi": "Titik Lokasi"},
)
fig_top_total.update_layout(yaxis={"categoryorder": "total ascending"})
simpan_figure(fig_top_total, "step5_top10_titik_total")

top10_rata = (
    df_normal.groupBy("titik_lokasi", "wilayah", "kecamatan")
    .agg(
        F.mean("volume_sampah_perhari").alias("rata_harian"),
        F.sum("volume_sampah_perhari").alias("total_volume"),
        F.count("*").alias("jumlah_hari"),
    )
    .filter(F.col("jumlah_hari") >= 30)  # hindari titik yang hanya muncul beberapa hari
    .orderBy(F.desc("rata_harian"))
    .limit(10)
    .toPandas()
)
print("Top 10 titik berdasarkan RATA-RATA harian (min 30 hari, is_outlier = 0):")
display(top10_rata)

fig_top_rata = px.bar(
    top10_rata,
    x="rata_harian",
    y="titik_lokasi",
    color="wilayah",
    orientation="h",
    title="Top 10 Titik Lokasi Berdasarkan Rata-rata Harian (min 30 hari, is_outlier = 0)",
    labels={"rata_harian": "Rata-rata harian (m³)", "titik_lokasi": "Titik Lokasi"},
)
fig_top_rata.update_layout(yaxis={"categoryorder": "total ascending"})
simpan_figure(fig_top_rata, "step5_top10_titik_rata")

Top 10 titik berdasarkan TOTAL volume (is_outlier = 0):


,titik_lokasi,wilayah,kecamatan,total_volume,rata_harian,jumlah_hari
0,KALI / SUNGAI CILIWUNG EMPLASEMENT PA MANGGARA...,JAKARTA PUSAT,MENTENG,15748.5000,20.242288,778
1,SEKATAN HDPE KALI SODETAN SEKRETARIS (JLN. TAN...,JAKARTA BARAT,GROGOL PETAMBURAN,10760.5000,15.663028,687
2,BANJIR KANAL BARAT SEASONS CITY (JLN. INSPEKSI...,JAKARTA BARAT,TAMBORA,9849.0000,20.691176,476
3,KALI / SUNGAI BKB CILIWUNG PETAMBURAN,JAKARTA PUSAT,TANAH ABANG,6778.5000,8.690385,780
4,KALI CAGAK,JAKARTA UTARA,PENJARINGAN,6531.1940,7.784498,839
5,KALI CENGKARENG DRAIN JLN. RAYA BASMOL/TANGGUL,JAKARTA BARAT,KEMBANGAN,6513.1035,8.777768,742
6,WADUK PLUIT,JAKARTA UTARA,PENJARINGAN,6362.6933,7.703019,826
7,KALI / SUNGAI SENTIONG P2SC S/D MARTO,JAKARTA PUSAT,KEMAYORAN,6323.9745,7.186335,880
8,BKB RPTRA KALIJODO,JAKARTA BARAT,TAMBORA,5698.5035,8.959911,636
9,KALI MUARA BARU,JAKARTA UTARA,PENJARINGAN,5300.1000,6.294656,842


Top 10 titik berdasarkan RATA-RATA harian (min 30 hari, is_outlier = 0):


,titik_lokasi,wilayah,kecamatan,rata_harian,total_volume,jumlah_hari
0,BANJIR KANAL BARAT SEASONS CITY (JLN. INSPEKSI...,JAKARTA BARAT,TAMBORA,26.438776,5182.0,196
1,BANJIR KANAL BARAT SEASONS CITY (JLN. INSPEKSI...,JAKARTA BARAT,TAMBORA,20.691176,9849.0,476
2,KALI / SUNGAI CILIWUNG EMPLASEMENT PA MANGGARA...,JAKARTA PUSAT,MENTENG,20.242288,15748.5,778
3,SEKATAN HDPE KALI SODETAN SEKRETARIS (JLN. TAN...,JAKARTA BARAT,GROGOL PETAMBURAN,15.663028,10760.5,687
4,KALI BUARAN (YAMAHAPINTU AIR UT),JAKARTA TIMUR,CAKUNG,13.250000,1802.0,136
5,SODETAN SEKRETARIS (POS POLISI CITRA LAND - SA...,JAKARTA BARAT,GROGOL PETAMBURAN,11.509091,1266.0,110
6,SARINGAN SAMPAH SEKRETARIS,JAKARTA BARAT,GROGOL PETAMBURAN,10.780822,787.0,73
7,PHB BELAKANG SQUARE WADUK RAWA KEPA (JLN. TOMA...,JAKARTA BARAT,GROGOL PETAMBURAN,10.416667,2375.0,228
8,PHB RAWA KEPA UJUNG RAWA KEPA RAYA (JLN. RAWA ...,JAKARTA BARAT,GROGOL PETAMBURAN,10.303419,2411.0,234
9,PHB KAMPUS UNIVERSITAS TARMUNEGARA MALL CITRA ...,JAKARTA BARAT,GROGOL PETAMBURAN,9.920168,2361.0,238


## 4. Sebaran Volume & Identifikasi Lonjakan / Outlier Ekstrem

In [6]:
kuantil = df.approxQuantile(
    "volume_sampah_perhari", [0.0, 0.25, 0.5, 0.75, 0.9, 0.99, 0.999, 1.0], 1e-5
)
df_kuantil = pd.DataFrame(
    {
        "persentil": ["min", "P25", "P50 (median)", "P75", "P90", "P99", "P99.9", "max"],
        "volume_m3": kuantil,
    }
)
display(df_kuantil)

n_nol = df.filter(F.col("volume_sampah_perhari") == 0).count()
print(f"Baris volume == 0 : {n_nol:,} ({n_nol / total_baris * 100:.2f}%)")

# Sampel terbatas untuk histogram & boxplot agar aman di RAM 1 GB.
SAMPEL_EDA = 20000
n_normal = df_normal.count()
fraksi = min(1.0, SAMPEL_EDA / max(n_normal, 1))
sampel_pd = (
    df_normal.select("wilayah", "volume_sampah_perhari")
    .sample(False, fraksi, seed=42)
    .limit(SAMPEL_EDA)
    .toPandas()
)

fig_hist = px.histogram(
    sampel_pd,
    x="volume_sampah_perhari",
    nbins=50,
    title="Distribusi Volume Pengangkutan Harian (sampel 20.000 baris, is_outlier = 0)",
    labels={"volume_sampah_perhari": "Volume harian (m³)"},
)
simpan_figure(fig_hist, "step5_distribusi_volume")

fig_box = px.box(
    sampel_pd,
    x="wilayah",
    y="volume_sampah_perhari",
    title="Boxplot Volume Harian per Wilayah (sampel 20.000 baris, is_outlier = 0)",
    labels={"wilayah": "Wilayah", "volume_sampah_perhari": "Volume harian (m³)"},
)
simpan_figure(fig_box, "step5_boxplot_wilayah")

# Ringkasan lonjakan ekstrem (> 500 m³) untuk audit kualitas data
lonjakan_ekstrem = (
    df.filter(F.col("volume_sampah_perhari") > 500)
    .groupBy("tahun", "bulan", "wilayah")
    .agg(
        F.count("*").alias("jumlah_kejadian"),
        F.max("volume_sampah_perhari").alias("maks_volume"),
    )
    .orderBy(F.desc("maks_volume"))
    .toPandas()
)
print("Ringkasan lonjakan ekstrem (> 500 m³):")
display(lonjakan_ekstrem)

,persentil,volume_m3
0,min,0.000000e+00
1,P25,0.000000e+00
2,P50 (median),5.000000e-01
3,P75,2.000000e+00
4,P90,4.480000e+00
5,P99,1.500000e+01
6,P99.9,6.552000e+01
7,max,1.562500e+08


Baris volume == 0 : 377,926 (35.37%)


Ringkasan lonjakan ekstrem (> 500 m³):


,tahun,bulan,wilayah,jumlah_kejadian,maks_volume
0,2018,7,JAKARTA UTARA,72,156249999.0
1,2018,2,JAKARTA PUSAT,2,3300.0
2,2018,9,JAKARTA PUSAT,1,2520.0
3,2020,9,JAKARTA PUSAT,1,2358.0
4,2020,10,JAKARTA PUSAT,1,2046.0
5,2018,5,JAKARTA PUSAT,2,2010.0
6,2018,6,JAKARTA PUSAT,2,1926.0
7,2020,4,JAKARTA BARAT,1,1192.0
8,2021,10,JAKARTA PUSAT,2,1176.0
9,2018,11,JAKARTA PUSAT,2,892.0


## 5. Rangkuman Jawaban Pertanyaan Analitik (1, 2, dan 4)

Seluruh angka di bawah berasal langsung dari hasil eksekusi sel 1–4 pada `1.068.562` baris bersih (`is_outlier == 0`: `1.067.494` baris normal, `is_outlier == 1`: `1.068` baris atau `0,100%` di atas P99,9 = `65,52 m³`).

1. **Pertanyaan Analitik 1 (Perubahan volume menurut waktu & wilayah):**
   - **Antar tahun (data normal `is_outlier = 0`):** total volume terangkut meningkat dari `366.811,48 m³` (`2018`, `251.490` baris, rata-rata `1,46 m³/baris`) menjadi `760.925,36 m³` (`2020`, `290.301` baris, rata-rata `2,63 m³/baris`) dan `789.941,13 m³` (`2021`, `526.771` baris, rata-rata `1,50 m³/baris`).
   - **Bulan puncak vs terendah (normal):** puncak total bulanan terjadi pada **Januari 2021** (`150.173,47 m³`), sedangkan terendah pada **Januari 2018** (`11.276,50 m³`).
   - **Antar wilayah (normal):** **Jakarta Barat** memiliki total volume tertinggi (`670.378,59 m³`, `1.074` titik), diikuti Jakarta Selatan (`386.779,46 m³`), Jakarta Pusat (`335.901,22 m³`), Jakarta Utara (`269.020,82 m³`), dan Jakarta Timur (`255.597,89 m³`). Namun dari sisi **intensitas rata-rata per observasi**, **Jakarta Pusat** tertinggi (`2,85 m³`).

2. **Pertanyaan Analitik 2 (Pola antar periode & anomali data):**
   - **Musim hujan vs kemarau:** rata-rata pengangkutan pada **musim hujan (Okt–Apr)** sebesar **`1,92 m³`** (`1.179.272 m³`), sekitar **17,4% lebih tinggi** dibandingkan **musim kemarau (Mei–Sep)** (`1,63 m³`, `738.406 m³`).
   - **Hari dalam minggu:** volume rata-rata memuncak di pertengahan minggu pada **Selasa** (`1,91 m³`) dan **Rabu** (`1,91 m³`), serta paling rendah pada **Jumat** (`1,71 m³`) dan **Minggu** (`1,72 m³`).
   - **Karakteristik sebaran & anomali:** `35,37%` baris bernilai `0 m³` (median `0,50 m³`, P90 `4,48 m³`, P99 `15,00 m³`). Ditemukan `95` kejadian ekstrem di atas `500 m³`, didominasi **72 kejadian pada Juli 2018 di Jakarta Utara** hingga **`156.249.999,0 m³`** yang mendistorsi total mentah 2018 menjadi `4,74 miliar m³` jika tidak dipisahkan.

3. **Pertanyaan Analitik 4 (Wilayah & titik lokasi dengan volume tertinggi):**
   - **Wilayah tertinggi:** **Jakarta Barat** secara total (`670.378,59 m³`) dan **Jakarta Pusat** secara rata-rata per baris (`2,85 m³`).
   - **Titik lokasi tertinggi (total normal):** `KALI / SUNGAI CILIWUNG EMPLASEMENT PA MANGGARAI` (Menteng, Jakarta Pusat) sebesar **`15.748,50 m³`** (`778` hari, rata-rata `20,24 m³/hari`), disusul `SEKATAN HDPE KALI SODETAN SEKRETARIS` (`10.760,50 m³`) dan `BANJIR KANAL BARAT SEASONS CITY` (`9.849,00 m³`).